# Web Research assistant
Research assistant that will research on a given task
1. Research assistant agent searches the web and summarizes results
2. Planner agent as the model for helpful search terms
2. Run websearch for the above terms
3. Ask the model to generate a report from the web search results

In [1]:
from agents import Agent, WebSearchTool, trace, Runner, gen_trace_id, function_tool
from agents.model_settings import ModelSettings
from pydantic import BaseModel, Field
from dotenv import load_dotenv
import asyncio
import os
from typing import Dict
from IPython.display import display, Markdown

In [2]:
load_dotenv(override=True)

True

## 1. Research Assistant agent


In [6]:
INSTRUCTIONS = "You are a research assistant. Given a search term, you search the web for that term and \
produce a concise summary of the results. The summary must 2-3 paragraphs and less than 300 \
words. Capture the main points. Write succintly, no need to have complete sentences or good \
grammar. This will be consumed by someone synthesizing a report, so it's vital you capture the \
essence and ignore any fluff. Do not include any additional commentary other than the summary itself."

search_agent = Agent(
    name="Search agent",
    instructions=INSTRUCTIONS,
    tools=[WebSearchTool(search_context_size="low")],
    model="gpt-4o-mini",
    model_settings=ModelSettings(tool_choice="required"),
)

In [7]:
# test research assistant agent

message = "Latest AI Agent frameworks in 2025"

with trace("Search"):
    result = await Runner.run(search_agent, message)

display(Markdown(result.final_output))

In 2025, several AI agent frameworks have emerged, each offering unique capabilities:

- **OpenAI Agents SDK**: Released in March 2025, this lightweight Python framework enables the creation of multi-agent workflows with comprehensive tracing and guardrails. It is compatible with over 100 different large language models (LLMs). ([jlcnews.com](https://www.jlcnews.com/post/the-best-ai-agents-in-2025-tools-frameworks-and-platforms-compared?utm_source=openai))

- **Google Agent Development Kit (ADK)**: Announced in April 2025, Google's ADK is a modular framework that integrates seamlessly with the Google ecosystem, including Gemini and Vertex AI. It supports hierarchical agent compositions and requires minimal code for efficient development. ([jlcnews.com](https://www.jlcnews.com/post/the-best-ai-agents-in-2025-tools-frameworks-and-platforms-compared?utm_source=openai))

- **Agent Lightning**: Introduced in August 2025, Agent Lightning is a flexible framework that facilitates reinforcement learning-based training of LLMs for any AI agent. It decouples agent execution from training, allowing integration with existing agents developed through various methods. ([arxiv.org](https://arxiv.org/abs/2508.03680?utm_source=openai))

- **Cognitive Kernel-Pro**: Launched in August 2025, this open-source framework is designed to democratize the development and evaluation of advanced AI agents. It focuses on curating high-quality training data and enhancing agent robustness and performance. ([arxiv.org](https://arxiv.org/abs/2508.00414?utm_source=openai))

- **Manus**: Released in March 2025, Manus is an autonomous AI agent developed by Butterfly Effect Pte. Ltd., later acquired by Meta Platforms. It is designed to independently carry out complex real-world tasks without direct or continuous human guidance. ([en.wikipedia.org](https://en.wikipedia.org/wiki/Manus_%28AI_agent%29?utm_source=openai))

These frameworks reflect the rapid advancements in AI agent development, offering diverse tools and platforms for building intelligent, autonomous systems. 

## 2. Planner Agent asks model for helpful search terms
Ask model for what to search on and use Structured Outputs

In [3]:

HOW_MANY_SEARCHES = 3

INSTRUCTIONS = f"You are a helpful research assistant. Given a query, come up with a set of web searches \
to perform to best answer the query. Output {HOW_MANY_SEARCHES} terms to query for."

# Use Pydantic to define the Schema of response which is known as "Structured Outputs"

class WebSearchItem(BaseModel):
    reason: str = Field(description="Your reasoning for why this search is important to the query.")
    query: str = Field(description="The search term to use for the web search.")


class WebSearchPlan(BaseModel):
    searches: list[WebSearchItem] = Field(description="A list of web searches to perform to best answer the query.")

planner_agent = Agent(
    name="Planner Agent",
    instructions=INSTRUCTIONS,
    model="gpt-4o-mini",
    output_type=WebSearchPlan
)

In [4]:
research_subject = "Latest AI Agent frameworks in 2025"

with trace("Search"):
    result = await Runner.run(planner_agent, research_subject)
    print(result.final_output)

searches=[WebSearchItem(reason='To find recent information and updates on AI agent frameworks in 2025.', query='latest AI agent frameworks 2025'), WebSearchItem(reason='To explore specific technologies or platforms that are emerging for AI agents in 2025.', query='top AI agent technologies 2025'), WebSearchItem(reason='To gather insights from industry experts on the trends and developments in AI agent frameworks for 2025.', query='AI agent frameworks trends 2025')]


## 3. Senior researcher agent writes a nice report
Writer agent is provided with the search results and requested for synthesize and format the report


In [5]:
INSTRUCTIONS = (
    "You are a senior researcher tasked with writing a cohesive report for a research query. "
    "You will be provided with the original query, and some initial research done by a research assistant.\n"
    "You should first come up with an outline for the report that describes the structure and "
    "flow of the report. Then, generate the report and return that as your final output.\n"
    "The final output should be in markdown format, and it should be lengthy and detailed. Aim "
    "for 5-10 pages of content, at least 1000 words."
)


class ReportData(BaseModel):
    short_summary: str = Field(description="A short 2-3 sentence summary of the findings.")

    markdown_report: str = Field(description="The final report")

    follow_up_questions: list[str] = Field(description="Suggested topics to research further")


writer_agent = Agent(
    name="WriterAgent",
    instructions=INSTRUCTIONS,
    model="gpt-4o-mini",
    output_type=ReportData,
)

# Functions to plan, perform and search using the agents created above

In [9]:
async def plan_searches(query: str):
    """ Use the planner_agent to plan which searches to run for the query """
    print("Planning searches...")
    result = await Runner.run(planner_agent, f"Query: {query}")
    print(f"Will perform {len(result.final_output.searches)} searches")
    return result.final_output


async def perform_searches(search_plan: WebSearchPlan):
    """ Call search() for each item in the search plan """
    print("Searching...")
    tasks = [asyncio.create_task(search(item)) for item in search_plan.searches]
    results = await asyncio.gather(*tasks)
    return results;

async def search(item: WebSearchItem):
    """ Use the search agent to run a web search for each item in the search plan """
    input = f"Search term: {item.query}\nReason for searching: {item.reason}"
    result = await Runner.run(search_agent, input)
    return result.final_output


async def write_report(query: str, search_results: list[str]):
    """ Use the writer agent to write a report based on the search results"""
    print("Thinking about report...")
    input = f"Original query: {query}\nSummarized search results: {search_results}"
    result = await Runner.run(writer_agent, input)
    print("Finished writing report")
    return result.final_output

## Put everything together

In [10]:
query ="Latest AI Agent frameworks in 2025"

with trace("Research trace"):
    print("Starting research...")
    search_plan = await plan_searches(query)
    search_results = await perform_searches(search_plan)
    report = await write_report(query, search_results)
    print("Hooray!")

Starting research...
Planning searches...
Will perform 3 searches
Searching...
Thinking about report...
Finished writing report
Hooray!


In [11]:
display(Markdown(report.markdown_report))

# Latest AI Agent Frameworks in 2025

The year 2025 marks a pivotal moment in the development of artificial intelligence (AI) frameworks, particularly those focused on autonomous agents. This report explores several of the latest AI agent frameworks that have emerged, their unique features, and the trends driving their adoption across different sectors. We will examine individual frameworks, their capabilities, and how they collectively represent a maturation of AI technologies.

## 1. Introduction  
The proliferation of AI agent frameworks in 2025 can be attributed to several factors, including advancements in machine learning, the increasing demand for automation, and the need for efficient workflows across various industries. As organizations strive to enhance productivity and streamline operations, AI agents have become central to achieving these goals. This report presents an overview of the most notable AI agent frameworks, highlighting their features and intended applications.

## 2. Overview of AI Agent Frameworks  
The following frameworks have garnered attention due to their capabilities in supporting the development and deployment of autonomous agents:

### 2.1 OpenAI Agents SDK  
Released in March 2025, the OpenAI Agents SDK is a lightweight Python framework designed for creating multi-agent workflows. It features comprehensive tracing and guardrails, which help developers maintain oversight while leveraging AI capabilities. With compatibility across over 100 different large language models, this framework provides ample versatility for various AI applications.

### 2.2 Google Project Mariner  
Launched in May 2025, Google’s Project Mariner serves as a research prototype aimed at automating web-based tasks that users commonly face, such as online shopping and information retrieval. By delegating routine tasks to AI agents, Project Mariner seeks to enhance user productivity, allowing individuals to focus on more complex activities.

### 2.3 Replit’s Agent 3  
Introduced in late 2025, Replit’s Agent 3 represents a significant advancement in autonomous AI agents. This platform enables agents to self-operate for extended periods, test and debug code autonomously, and even create other agents. This level of independence facilitates rapid software development through natural language prompts, making it a powerful tool for developers.

### 2.4 Google Antigravity  
Announced in November 2025, Google Antigravity is an integrated development environment (IDE) built on Google’s Gemini 3 models. This framework introduces the "agent-first" paradigm to coding workflows, enabling developers to delegate complex tasks to autonomous AI agents and support asynchronous coding.

### 2.5 Oracle Agentic AI Platform  
Focusing on the retail banking sector, Oracle unveiled its agentic AI platform tailored for enhancing automation and personalization across digital and in-person banking services. This platform features a suite of AI applications designed to maintain human oversight and ethical governance, improving operational efficiency within financial institutions.

### 2.6 Microsoft Agent 365  
Launched in late 2025, Microsoft’s Agent 365 framework automates tasks and streamlines workflows across digital platforms, positioning AI agents at the forefront of productivity solutions for users.

### 2.7 LangChain’s LangGraph  
As a response to the need for better agent orchestration, LangChain shifted to recommend LangGraph over its previous chain-based design. This updated framework is optimized for handling complex workflows, which is vital as organizations adopt AI agents for more intricate tasks.

### 2.8 Polymorphic Combinatorial Frameworks (PCF)  
While not a specific framework, the emergence of Polymorphic Combinatorial Frameworks integrates large language models with mathematical structures. This design enables the creation of adaptable AI agents that can perform real-time behavioral adjustments based on environmental changes, enhancing their application across sectors.

## 3. Key Trends in AI Agent Frameworks  
As noted, the frameworks listed are emblematic of larger trends shaping the AI agent landscape in 2025. These include:

### 3.1 Increased Adaptability  
The integration of modular architectures and real-time behavioral adjustments allows for the adaptive deployment of AI agents across various use cases. This flexibility dramatically improves the usability of AI technology in dynamic environments, such as customer service and healthcare.

### 3.2 Consolidation of Offerings  
The ongoing consolidation among leading players like Microsoft and LangChain indicates a trend toward refined, robust frameworks designed to handle the demands of complex, multi-faceted workflows.

### 3.3 Enhanced Security and Governance  
The shift from traditional architectures towards agent-based AI systems tends to enhance security and simplify maintenance. By preserving existing access controls and reducing centralized data storage, enterprises can better navigate the regulatory environment that governs data usage and privacy.

### 3.4 Outcome as Agentic Solution (OaAS) Model  
The rise of OaAS signifies a transformation in business operations. This model allows AI agents to autonomously deliver outcomes for customers, making vendors accountable for specific results rather than simply providing tools. This approach is anticipated to revolutionize enterprise interactions by fostering auditable, orchestrated execution workflows.

## 4. Conclusion  
In summary, the advancements in AI agent frameworks during 2025 reflect significant progress in the field. Notable frameworks such as OpenAI Agents SDK, Google Project Mariner, and Replit’s Agent 3 provide robust platforms for developing autonomous systems that enhance productivity and streamline workflows. Additionally, the emerging trends highlight a concerted effort towards security, adaptability, and outcome-focus, indicating a bright future for AI in transforming how we interact with technology. As these frameworks continue to evolve, they will undoubtedly redefine standards across various industries, facilitating an era where AI agents play an integral role in daily operations.

## 5. Follow Up Questions  
1. What are the ethical considerations in deploying AI agents across critical sectors?  
2. How will AI agent frameworks evolve beyond 2025?  
3. What role do regulatory frameworks play in the development of AI agents?  
4. How can organizations ensure the security of their AI agent systems?  
5. What impact do these frameworks have on the workforce and employment?  
6. How do user interfaces play a role in the interaction between humans and AI agents?  
7. What are the best practices for integrating AI agents into existing systems?  

This comprehensive analysis serves as a foundational resource for understanding the latest developments in AI agent frameworks as of 2025, inviting further exploration into their implications and future trajectory.